[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/04.%20The%20FCFS%20Berth%20Scheduling%20Problem/P4-Tier-2_executed.ipynb)

---

# 4. The FCFS Berth Scheduling Problem

## Tier 2: Priority-Based FCFS Heuristic

### Purpose and inputs

This notebook implements a priority-based first-come, first-served (FCFS) heuristic for assigning arriving vessels to compatible berths. The inputs are vessel arrival and processing times, commercial and urgency attributes, vessel and berth dimensions, priority weights, and an FCFS time-window width. The outputs are a berth schedule and descriptive performance metrics, including waiting time, makespan, berth utilization, and FCFS compliance.

A heuristic is a rule-based procedure that produces a feasible schedule without guaranteeing that it is globally optimal. Here, arrival times establish the base order; vessels in the same arrival window may be reordered by a weighted priority score. Berth assignments then respect length and draft compatibility and use the compatible berth with the earliest availability.

### Model assumptions and scope

- Arrival times and processing times are known and deterministic, and all times are measured in hours.
- Each berth can serve at most one vessel at a time. Processing is non-preemptive: once service starts, it continues until completion.
- Vessel length and draft requirements are represented by `max_length` and `max_draft`; a berth is compatible when both requirements do not exceed its capacities.
- The vessel dimensions are not separately specified in the example, so the data class defaults of 400 m and 16 m apply to every example vessel. Consequently, only a berth meeting both of those requirements can serve them.
- Priority combines normalized TEU capacity, revenue, customer tier, and urgency. The weights are nonnegative and sum to one.
- The FCFS window is anchored at the earliest arrival in each window. A vessel joins that window only if its arrival is no later than the anchor plus the window width; windows do not grow by chaining successive arrivals.
- The model does not include tides, setup times, uncertain processing durations, vessel movements, or changes to arrivals after scheduling.

### Priority score

The priority score is a weighted sum of four dimensionless factors:

$$
P_i = w_T \frac{T_i}{T_{\max}} + w_R \frac{R_i}{R_{\max}} + w_C \frac{3-C_i}{2} + w_U U_i.
$$

Here, $i$ identifies a vessel; $T_i$ is its TEU capacity; $R_i$ is its revenue in thousands of dollars; $C_i \in \{1,2,3\}$ is its customer tier, with tier 1 receiving the largest tier score; and $U_i \in [0,1]$ is its urgency factor. The normalization values $T_{\max}$ and $R_{\max}$ are the largest TEU capacity and revenue in the vessel set. The weights $(w_T,w_R,w_C,w_U)$ correspond to those four factors. A larger $P_i$ places a vessel earlier in its window's priority order; it does not by itself guarantee an earlier service start, because berth availability and vessel arrival time still apply.

### Scheduling procedure

1. Sort vessels by arrival time and partition them into anchored FCFS windows.
2. Calculate each vessel's priority score and sort each window in descending score order.
3. For each vessel in that order, select a compatible berth with the earliest availability, start service no earlier than both the vessel's arrival and berth availability, and update the berth's availability.
4. Summarize the resulting schedule. Waiting time is service start minus arrival; makespan is the latest service completion time.

The heuristic preserves chronological window order, but permits priority-based reordering within each window. Because berths are chosen greedily, the procedure does not guarantee a minimum total waiting time or makespan.

## Mathematical description

Let $V$ be the set of vessels and $B$ the set of berths. Vessel $i \in V$ has arrival time $a_i$, processing time $p_i$, and priority score $P_i$. Berth $b \in B$ has availability time $A_b$ during the construction of the schedule. Define the compatible-berth set

$$
B_i = \{b \in B : L_i \leq L_b,\; D_i \leq D_b\},
$$

where $L_i,D_i$ are the vessel's length and draft requirements, and $L_b,D_b$ are the berth's corresponding capacities. The code represents these quantities using the vessel and berth fields `max_length` and `max_draft`.

For each vessel, the heuristic chooses a berth from $B_i$ with the smallest current availability, breaking ties by the order of berths in the input list. If that berth is $b$, its service start and completion times are

$$
s_i = \max(a_i,A_b), \qquad c_i = s_i+p_i,
$$

and its waiting time is

$$
W_i = s_i-a_i = \max(0,A_b-a_i).
$$

After assigning vessel $i$, the algorithm sets $A_b \leftarrow c_i$. The other berths retain their availability times. These equations correspond to `start_time`, `end_time`, `waiting_time`, and `available_time` in the implementation.

For an anchored window whose first arrival is $a_0$, vessels satisfying $a_i \leq a_0+\Delta$ belong to that window, where $\Delta$ is `fcfs_window`. Within each window, vessels are processed in descending $P_i$ order. The schedule is therefore a greedy constructive heuristic, not an optimization model with an optimality guarantee.

With $n=|V|$ vessels and $m=|B|$ berths, score calculation and window construction take $O(n)$ time after arrival sorting; sorting arrivals and priority orders takes at most $O(n\log n)$; and checking berths for each vessel takes $O(nm)$. The overall construction is therefore $O(n\log n+nm)$, excluding result visualization. The schedule uses $O(n+m)$ storage. These bounds describe this implementation, not a guarantee about the quality of its schedule.

In [ ]:
import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

print("Required packages imported successfully.")

In [ ]:
@dataclass
class EnhancedVessel:
    """Vessel data used by the priority-based scheduling heuristic."""

    id: int
    arrival_time: float
    processing_time: float
    teu_capacity: int
    revenue: float  # Thousands of dollars.
    customer_tier: int  # 1 = premium, 2 = standard, 3 = budget.
    urgency_factor: float  # A score in [0, 1].
    max_length: float = 400.0
    max_draft: float = 16.0
    priority_score: float = field(default=0.0, init=False)

    def calculate_priority_score(
        self,
        weights: Tuple[float, float, float, float],
        max_teu: int,
        max_revenue: float,
    ) -> float:
        """Calculate the weighted score using maxima from the vessel set."""
        w_teu, w_revenue, w_tier, w_urgency = weights
        teu_score = self.teu_capacity / max_teu
        revenue_score = self.revenue / max_revenue
        tier_score = (3 - self.customer_tier) / 2.0

        return (
            w_teu * teu_score
            + w_revenue * revenue_score
            + w_tier * tier_score
            + w_urgency * self.urgency_factor
        )


@dataclass
class Berth:
    """Berth capacities and the next time at which it is available."""

    id: int
    max_length: float
    max_draft: float
    available_time: float = 0.0


@dataclass
class ScheduleAssignment:
    """Service interval and waiting time for one vessel assignment."""

    vessel_id: int
    berth_id: int
    start_time: float
    end_time: float
    waiting_time: float
    priority_score: float


print("Vessel, berth, and assignment data structures are ready.")

In [ ]:
class PriorityBasedFCFSScheduler:
    """Construct a priority-based FCFS berth schedule using a greedy rule."""

    def __init__(
        self,
        vessels: List[EnhancedVessel],
        berths: List[Berth],
        priority_weights: Tuple[float, float, float, float] = (
            0.4,
            0.3,
            0.2,
            0.1,
        ),
        fcfs_window: float = 2.0,
    ) -> None:
        if not vessels:
            raise ValueError("At least one vessel is required.")
        if not berths:
            raise ValueError("At least one berth is required.")
        if len(priority_weights) != 4 or any(weight < 0 for weight in priority_weights):
            raise ValueError("Provide four nonnegative priority weights.")
        if not math.isclose(sum(priority_weights), 1.0, rel_tol=0.0, abs_tol=1e-9):
            raise ValueError("Priority weights must sum to 1.")
        if fcfs_window < 0:
            raise ValueError("The FCFS window must be nonnegative.")
        if len({vessel.id for vessel in vessels}) != len(vessels):
            raise ValueError("Vessel IDs must be unique.")
        if len({berth.id for berth in berths}) != len(berths):
            raise ValueError("Berth IDs must be unique.")

        for vessel in vessels:
            if vessel.arrival_time < 0 or vessel.processing_time <= 0:
                raise ValueError("Arrival times must be nonnegative and processing times positive.")
            if vessel.teu_capacity <= 0 or vessel.revenue <= 0:
                raise ValueError("TEU capacity and revenue must be positive.")
            if vessel.customer_tier not in (1, 2, 3):
                raise ValueError("Customer tier must be 1, 2, or 3.")
            if not 0.0 <= vessel.urgency_factor <= 1.0:
                raise ValueError("Urgency factors must be in [0, 1].")
        if any(berth.max_length < 0 or berth.max_draft < 0 for berth in berths):
            raise ValueError("Berth capacities must be nonnegative.")

        self.vessels = list(vessels)
        self.berths = list(berths)
        self.priority_weights = priority_weights
        self.fcfs_window = fcfs_window
        self.vessels_by_arrival = sorted(
            self.vessels, key=lambda vessel: (vessel.arrival_time, vessel.id)
        )
        self._calculate_priority_scores()
        self.schedule: List[ScheduleAssignment] = []
        self.performance_metrics: Dict[str, object] = {}

    def _calculate_priority_scores(self) -> None:
        """Normalize TEU and revenue against maxima in this vessel set."""
        max_teu = max(vessel.teu_capacity for vessel in self.vessels)
        max_revenue = max(vessel.revenue for vessel in self.vessels)
        for vessel in self.vessels:
            vessel.priority_score = vessel.calculate_priority_score(
                self.priority_weights, max_teu, max_revenue
            )

    def _create_fcfs_windows(self) -> List[List[EnhancedVessel]]:
        """Partition arrival-sorted vessels into windows anchored at first arrival."""
        windows: List[List[EnhancedVessel]] = []
        current_window: List[EnhancedVessel] = []
        window_start_time: Optional[float] = None

        for vessel in self.vessels_by_arrival:
            if window_start_time is None:
                current_window = [vessel]
                window_start_time = vessel.arrival_time
            elif vessel.arrival_time <= window_start_time + self.fcfs_window:
                current_window.append(vessel)
            else:
                windows.append(current_window)
                current_window = [vessel]
                window_start_time = vessel.arrival_time

        if current_window:
            windows.append(current_window)
        return windows

    @staticmethod
    def _is_compatible(vessel: EnhancedVessel, berth: Berth) -> bool:
        """Check whether berth length and draft capacities meet vessel needs."""
        return (
            vessel.max_length <= berth.max_length
            and vessel.max_draft <= berth.max_draft
        )

    def _find_best_berth_for_vessel(
        self, vessel: EnhancedVessel, current_time: float
    ) -> Optional[Berth]:
        """Choose a compatible berth with the earliest effective availability."""
        compatible_berths = [
            berth for berth in self.berths if self._is_compatible(vessel, berth)
        ]
        if not compatible_berths:
            return None
        return min(
            compatible_berths,
            key=lambda berth: max(berth.available_time, current_time),
        )

    def schedule_vessels(self) -> List[ScheduleAssignment]:
        """Schedule each window by priority and update berth availability."""
        for berth in self.berths:
            berth.available_time = 0.0

        schedule: List[ScheduleAssignment] = []
        for window_vessels in self._create_fcfs_windows():
            window_vessels_sorted = sorted(
                window_vessels,
                key=lambda vessel: (-vessel.priority_score, vessel.arrival_time, vessel.id),
            )

            for vessel in window_vessels_sorted:
                compatible_berths = [
                    berth
                    for berth in self.berths
                    if self._is_compatible(vessel, berth)
                ]
                if not compatible_berths:
                    continue

                earliest_start = max(
                    vessel.arrival_time,
                    min(berth.available_time for berth in compatible_berths),
                )
                best_berth = self._find_best_berth_for_vessel(
                    vessel, earliest_start
                )
                if best_berth is None:
                    continue

                start_time = max(vessel.arrival_time, best_berth.available_time)
                end_time = start_time + vessel.processing_time
                assignment = ScheduleAssignment(
                    vessel_id=vessel.id,
                    berth_id=best_berth.id,
                    start_time=start_time,
                    end_time=end_time,
                    waiting_time=start_time - vessel.arrival_time,
                    priority_score=vessel.priority_score,
                )
                schedule.append(assignment)
                best_berth.available_time = end_time

        self.schedule = schedule
        self._calculate_performance_metrics()
        return schedule

    def _calculate_performance_metrics(self) -> None:
        """Calculate schedule metrics and compare start order across windows."""
        assigned_ids = {assignment.vessel_id for assignment in self.schedule}
        vessel_by_id = {vessel.id: vessel for vessel in self.vessels}
        makespan = max(
            (assignment.end_time for assignment in self.schedule), default=0.0
        )

        berth_utilization: Dict[int, float] = {}
        for berth in self.berths:
            processing_time = sum(
                assignment.end_time - assignment.start_time
                for assignment in self.schedule
                if assignment.berth_id == berth.id
            )
            berth_utilization[berth.id] = (
                processing_time / makespan * 100.0 if makespan > 0 else 0.0
            )

        windows = self._create_fcfs_windows()
        window_by_vessel = {
            vessel.id: window_index
            for window_index, window in enumerate(windows)
            for vessel in window
        }
        comparable_pairs = 0
        fcfs_violations = 0
        for first in self.schedule:
            for second in self.schedule:
                first_vessel = vessel_by_id[first.vessel_id]
                second_vessel = vessel_by_id[second.vessel_id]
                if (
                    window_by_vessel[first.vessel_id]
                    < window_by_vessel[second.vessel_id]
                ):
                    comparable_pairs += 1
                    if second.start_time < first.start_time:
                        fcfs_violations += 1

        total_waiting = sum(
            assignment.waiting_time for assignment in self.schedule
        )
        assigned_count = len(self.schedule)
        self.performance_metrics = {
            "total_waiting_time": total_waiting,
            "average_waiting_time": (
                total_waiting / assigned_count if assigned_count else 0.0
            ),
            "makespan": makespan,
            "berth_utilization": berth_utilization,
            "average_berth_utilization": (
                float(np.mean(list(berth_utilization.values())))
                if berth_utilization
                else 0.0
            ),
            "fcfs_violations": fcfs_violations,
            "fcfs_compliance_rate": (
                100.0 * (comparable_pairs - fcfs_violations) / comparable_pairs
                if comparable_pairs
                else 100.0
            ),
            "comparable_fcfs_pairs": comparable_pairs,
            "assigned_vessels": assigned_count,
            "unassigned_vessels": len(self.vessels) - len(assigned_ids),
        }

    def visualize_schedule(self) -> None:
        """Plot service intervals, waiting times, utilization, and windows."""
        if not self.schedule:
            print("No vessel could be assigned to a compatible berth.")
            return

        vessel_by_id = {vessel.id: vessel for vessel in self.vessels}
        berth_ids = [berth.id for berth in self.berths]
        berth_positions = {berth_id: index for index, berth_id in enumerate(berth_ids)}
        colors = plt.cm.Set3(np.linspace(0, 1, max(len(self.vessels), 1)))
        color_by_vessel = {
            vessel.id: colors[index]
            for index, vessel in enumerate(self.vessels)
        }

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        gantt_ax, waiting_ax, utilization_ax, window_ax = axes.flat

        for assignment in self.schedule:
            position = berth_positions[assignment.berth_id]
            duration = assignment.end_time - assignment.start_time
            gantt_ax.barh(
                position,
                duration,
                left=assignment.start_time,
                color=color_by_vessel[assignment.vessel_id],
                edgecolor="black",
            )
            gantt_ax.text(
                assignment.start_time + duration / 2,
                position,
                f"V{assignment.vessel_id}",
                ha="center",
                va="center",
                fontsize=8,
            )
        gantt_ax.set_yticks(range(len(berth_ids)))
        gantt_ax.set_yticklabels([str(berth_id) for berth_id in berth_ids])
        gantt_ax.set_xlabel("Time (hours)")
        gantt_ax.set_ylabel("Berth ID")
        gantt_ax.set_title("Berth Schedule")
        gantt_ax.grid(True, axis="x", alpha=0.3)

        waiting_ax.scatter(
            [assignment.priority_score for assignment in self.schedule],
            [assignment.waiting_time for assignment in self.schedule],
            color="steelblue",
            edgecolors="black",
        )
        for assignment in self.schedule:
            waiting_ax.annotate(
                f"V{assignment.vessel_id}",
                (assignment.priority_score, assignment.waiting_time),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )
        waiting_ax.set_xlabel("Priority score")
        waiting_ax.set_ylabel("Waiting time (hours)")
        waiting_ax.set_title("Priority Score and Waiting Time")
        waiting_ax.grid(True, alpha=0.3)

        utilization = self.performance_metrics["berth_utilization"]
        utilization_ax.bar(
            list(utilization), list(utilization.values()), color="lightblue", edgecolor="black"
        )
        utilization_ax.set_xlabel("Berth ID")
        utilization_ax.set_ylabel("Utilization (%)")
        utilization_ax.set_title("Berth Utilization")
        utilization_ax.grid(True, axis="y", alpha=0.3)

        windows = self._create_fcfs_windows()
        for window_index, window in enumerate(windows):
            assigned = [
                assignment
                for assignment in self.schedule
                if assignment.vessel_id in {vessel.id for vessel in window}
            ]
            window_ax.scatter(
                [vessel_by_id[item.vessel_id].arrival_time for item in assigned],
                [item.priority_score for item in assigned],
                label=f"Window {window_index + 1}",
                s=80,
                edgecolors="black",
            )
        window_ax.set_xlabel("Arrival time (hours)")
        window_ax.set_ylabel("Priority score")
        window_ax.set_title("Arrival Windows and Priority")
        if windows:
            window_ax.legend()
        window_ax.grid(True, alpha=0.3)

        fig.suptitle("Priority-Based FCFS Schedule Analysis")
        fig.tight_layout()
        plt.show()

    def print_detailed_results(self) -> None:
        """Print performance metrics and the assigned service intervals."""
        metrics = self.performance_metrics
        print("PRIORITY-BASED FCFS SCHEDULING RESULTS")
        print("=" * 48)
        print(f"Priority weights: {self.priority_weights}")
        print(f"FCFS window: {self.fcfs_window:.1f} hours")
        print(
            f"Vessels assigned: {metrics.get('assigned_vessels', 0)}/"
            f"{len(self.vessels)}"
        )
        print(f"Vessels without a compatible berth: {metrics.get('unassigned_vessels', 0)}")
        print(f"Total waiting time: {metrics.get('total_waiting_time', 0.0):.1f} hours")
        print(f"Average waiting time: {metrics.get('average_waiting_time', 0.0):.1f} hours")
        print(f"Makespan: {metrics.get('makespan', 0.0):.1f} hours")
        print(
            "Average berth utilization: "
            f"{metrics.get('average_berth_utilization', 0.0):.1f}%"
        )
        print(
            "FCFS compliance across different windows: "
            f"{metrics.get('fcfs_compliance_rate', 100.0):.1f}%"
        )
        print(
            f"Comparable cross-window pairs: "
            f"{metrics.get('comparable_fcfs_pairs', 0)}; "
            f"violations: {metrics.get('fcfs_violations', 0)}"
        )

        print("\nAssigned vessel schedule (ordered by service start):")
        print("Vessel  Priority  Berth  Start  End  Wait  TEU  Tier")
        for assignment in sorted(self.schedule, key=lambda item: (item.start_time, item.vessel_id)):
            vessel = next(
                vessel for vessel in self.vessels if vessel.id == assignment.vessel_id
            )
            print(
                f"{assignment.vessel_id:<7} "
                f"{assignment.priority_score:<9.3f} "
                f"{assignment.berth_id:<6} "
                f"{assignment.start_time:<5.1f} "
                f"{assignment.end_time:<4.1f} "
                f"{assignment.waiting_time:<5.1f} "
                f"{vessel.teu_capacity:<4} "
                f"{vessel.customer_tier}"
            )


print("Priority-based FCFS scheduler is ready.")

## Example data

The example has five vessels and three berths. Revenue is measured in thousands of dollars, and all arrival and processing times are in hours. The commercial attributes and the priority weights are inputs to the heuristic; berth length and draft capacities determine physical compatibility.

In [ ]:
vessels = [
    EnhancedVessel(
        id=1, arrival_time=0.0, processing_time=8.0,
        teu_capacity=15000, revenue=100.0, customer_tier=1, urgency_factor=0.9
    ),
    EnhancedVessel(
        id=2, arrival_time=0.5, processing_time=4.0,
        teu_capacity=5000, revenue=30.0, customer_tier=2, urgency_factor=0.6
    ),
    EnhancedVessel(
        id=3, arrival_time=1.0, processing_time=12.0,
        teu_capacity=20000, revenue=150.0, customer_tier=1, urgency_factor=0.95
    ),
    EnhancedVessel(
        id=4, arrival_time=1.5, processing_time=3.0,
        teu_capacity=8000, revenue=60.0, customer_tier=2, urgency_factor=0.3
    ),
    EnhancedVessel(
        id=5, arrival_time=2.0, processing_time=6.0,
        teu_capacity=12000, revenue=80.0, customer_tier=1, urgency_factor=0.7
    ),
]

berths = [
    Berth(id=0, max_length=400, max_draft=16),
    Berth(id=1, max_length=350, max_draft=14),
    Berth(id=2, max_length=300, max_draft=12),
]

print(f"Vessels: {len(vessels)}; berths: {len(berths)}")
print("\nVessel data:")
for vessel in vessels:
    print(
        f"Vessel {vessel.id}: arrival={vessel.arrival_time:.1f} h, "
        f"processing={vessel.processing_time:.1f} h, "
        f"TEU={vessel.teu_capacity}, revenue=${vessel.revenue:.1f}K, "
        f"tier={vessel.customer_tier}, urgency={vessel.urgency_factor:.2f}, "
        f"length requirement={vessel.max_length:.0f} m, "
        f"draft requirement={vessel.max_draft:.0f} m"
    )

print("\nBerth capacities:")
for berth in berths:
    print(
        f"Berth {berth.id}: length capacity={berth.max_length:.0f} m, "
        f"draft capacity={berth.max_draft:.0f} m"
    )

## Run the scheduling heuristic

The weights below give the example's priority score. The window width determines which vessels may be reordered relative to one another: priority is applied only within each anchored window. Because the example vessels use the default length and draft requirements, berth compatibility is determined by those requirements and the listed capacities.

In [ ]:
priority_weights = (0.4, 0.3, 0.2, 0.1)
fcfs_window = 2.0

scheduler = PriorityBasedFCFSScheduler(
    vessels=vessels,
    berths=berths,
    priority_weights=priority_weights,
    fcfs_window=fcfs_window,
)
schedule = scheduler.schedule_vessels()

print(f"Priority scores calculated for {len(vessels)} vessels.")
print(f"Arrival-based FCFS windows created: {len(scheduler._create_fcfs_windows())}.")
print(f"Vessels assigned to a compatible berth: {len(schedule)}.")

## Read and verify the schedule

The summary reports metrics for the schedule produced by the heuristic; it does not compare them with an unverified target schedule. Utilization is each berth's total assigned processing time divided by the schedule makespan. FCFS compliance compares service start times for assigned vessel pairs that belong to different chronological windows. Within-window priority reordering is permitted and is not counted as a violation.

The assertions below check basic schedule consistency: every assignment refers to a known vessel and berth, service starts after arrival, waiting time agrees with arrival and start times, and the reported total waiting time equals the sum over assignments.

In [ ]:
scheduler.print_detailed_results()

vessel_by_id = {vessel.id: vessel for vessel in vessels}
berth_by_id = {berth.id: berth for berth in berths}

for assignment in schedule:
    vessel = vessel_by_id[assignment.vessel_id]
    berth = berth_by_id[assignment.berth_id]
    assert scheduler._is_compatible(vessel, berth)
    assert assignment.start_time >= vessel.arrival_time
    assert math.isclose(
        assignment.waiting_time,
        assignment.start_time - vessel.arrival_time,
        rel_tol=0.0,
        abs_tol=1e-9,
    )
    assert math.isclose(
        assignment.end_time - assignment.start_time,
        vessel.processing_time,
        rel_tol=0.0,
        abs_tol=1e-9,
    )

assert math.isclose(
    sum(assignment.waiting_time for assignment in schedule),
    scheduler.performance_metrics["total_waiting_time"],
    rel_tol=0.0,
    abs_tol=1e-9,
)
print("\nSchedule consistency checks passed.")

## Visualize the schedule and compare with strict FCFS

The plots show assigned service intervals, the relationship between priority and waiting time, utilization by berth, and the arrival windows. Color and labels distinguish vessels in the schedule plot; the window plot includes a legend.

For context, the following comparison constructs a strict-FCFS schedule by processing vessels in arrival order and applying the same earliest-available compatible-berth rule. This is a descriptive baseline for the same example, not a replacement scheduling method. Both schedules use independent berth-availability records so the comparison does not alter the priority-based schedule.

In [ ]:
scheduler.visualize_schedule()

strict_fcfs_waiting_times = []
strict_fcfs_berth_times = {berth.id: 0.0 for berth in berths}
strict_fcfs_assignments = []

for vessel in sorted(vessels, key=lambda item: (item.arrival_time, item.id)):
    compatible_berths = [
        berth
        for berth in berths
        if scheduler._is_compatible(vessel, berth)
    ]
    if not compatible_berths:
        continue

    best_berth = min(
        compatible_berths,
        key=lambda berth: max(
            strict_fcfs_berth_times[berth.id], vessel.arrival_time
        ),
    )
    start_time = max(
        strict_fcfs_berth_times[best_berth.id], vessel.arrival_time
    )
    waiting_time = start_time - vessel.arrival_time
    strict_fcfs_waiting_times.append(waiting_time)
    strict_fcfs_assignments.append((vessel.id, best_berth.id, start_time))
    strict_fcfs_berth_times[best_berth.id] = start_time + vessel.processing_time

strict_fcfs_total_wait = sum(strict_fcfs_waiting_times)
priority_total_wait = scheduler.performance_metrics["total_waiting_time"]
if strict_fcfs_total_wait > 0:
    waiting_time_change = (
        strict_fcfs_total_wait - priority_total_wait
    ) / strict_fcfs_total_wait * 100.0
    comparison_label = "reduction" if waiting_time_change >= 0 else "increase"
    print(
        f"Priority-based total waiting time: {priority_total_wait:.1f} hours"
    )
    print(f"Strict-FCFS total waiting time: {strict_fcfs_total_wait:.1f} hours")
    print(
        f"Relative waiting-time {comparison_label}: "
        f"{abs(waiting_time_change):.1f}%"
    )
else:
    print("Strict-FCFS total waiting time is zero; a relative change is undefined.")

print(
    "Priority-based FCFS compliance across different windows: "
    f"{scheduler.performance_metrics['fcfs_compliance_rate']:.1f}%"
)

## Sensitivity to priority weights

A sensitivity analysis changes the relative importance assigned to the four score components while keeping the vessel data, berths, and window width fixed. Each scenario constructs a fresh scheduler, so its berth-availability state is independent of the main example. The resulting schedules indicate whether these weight choices change the heuristic's ordering and performance for this particular data set; they do not establish a general ranking of weight strategies.

In [ ]:
weight_scenarios = [
    ("Balanced", (0.25, 0.25, 0.25, 0.25)),
    ("TEU-focused", (0.6, 0.2, 0.1, 0.1)),
    ("Revenue-focused", (0.2, 0.6, 0.1, 0.1)),
    ("Customer-tier-focused", (0.1, 0.1, 0.7, 0.1)),
    ("Urgency-focused", (0.1, 0.1, 0.1, 0.7)),
]

sensitivity_results = []
for scenario_name, weights in weight_scenarios:
    test_scheduler = PriorityBasedFCFSScheduler(
        vessels=vessels,
        berths=berths,
        priority_weights=weights,
        fcfs_window=fcfs_window,
    )
    test_scheduler.schedule_vessels()
    metrics = test_scheduler.performance_metrics
    sensitivity_results.append(
        {
            "scenario": scenario_name,
            "weights": weights,
            "total_waiting_time": metrics["total_waiting_time"],
            "average_waiting_time": metrics["average_waiting_time"],
            "makespan": metrics["makespan"],
            "order": tuple(
                assignment.vessel_id
                for assignment in sorted(
                    test_scheduler.schedule,
                    key=lambda item: (item.start_time, item.vessel_id),
                )
            ),
        }
    )

print("Priority-weight sensitivity results")
print("Scenario                 Weights                  Total wait  Average wait  Makespan")
for result in sensitivity_results:
    weights_text = ", ".join(f"{weight:.2f}" for weight in result["weights"])
    print(
        f"{result['scenario']:<24} "
        f"({weights_text})  "
        f"{result['total_waiting_time']:>10.1f}  "
        f"{result['average_waiting_time']:>12.1f}  "
        f"{result['makespan']:>7.1f}"
    )

scenario_names = [result["scenario"] for result in sensitivity_results]
total_waits = [result["total_waiting_time"] for result in sensitivity_results]
makespans = [result["makespan"] for result in sensitivity_results]
positions = np.arange(len(scenario_names))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(positions, total_waits, color="lightcoral", edgecolor="black")
axes[0].set_ylabel("Total waiting time (hours)")
axes[0].set_title("Total Waiting Time by Weight Scenario")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(scenario_names, rotation=35, ha="right")
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(positions, makespans, color="skyblue", edgecolor="black")
axes[1].set_ylabel("Makespan (hours)")
axes[1].set_title("Makespan by Weight Scenario")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(scenario_names, rotation=35, ha="right")
axes[1].grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

distinct_orders = {result["order"] for result in sensitivity_results}
distinct_waiting_totals = {
    round(result["total_waiting_time"], 9) for result in sensitivity_results
}
print(
    f"Distinct service-start orders across scenarios: {len(distinct_orders)} "
    f"out of {len(sensitivity_results)} scenarios."
)
print(
    f"Distinct total waiting-time values across scenarios: "
    f"{len(distinct_waiting_totals)}."
)

## Interpreting the heuristic

The priority score is a ranking device, not a direct measure of operational cost or revenue. A higher score can move a vessel ahead of another vessel in the same arrival window, while the vessel's arrival time and compatible berth availability still constrain its service start. The total waiting time and makespan are outcomes of the complete greedy scheduling procedure, not quantities directly minimized by an optimization solver.

### Strengths and limitations

- **Strengths:** The procedure is transparent, constructs a schedule quickly, incorporates several business attributes, and checks length and draft compatibility.
- **Limitations:** It has no optimality guarantee; greedy berth choices can affect later assignments; and scores depend on the chosen weights and normalization values. In this example, the default vessel requirements allow service only at a berth whose capacities meet both default dimensions, so the smaller listed berths cannot serve these vessels.
- **Interpretation of utilization:** Each berth's utilization is measured over the common makespan. The unweighted average includes berths that receive no assignments, so it describes average utilization across all listed berths rather than utilization only among used berths.
- **Interpretation of compliance:** The reported compliance rate is based on assigned vessel pairs in different chronological windows. It is 100% when there are no comparable cross-window pairs, because no such pair can violate the ordering rule.

The example illustrates how weighted priority and FCFS windows interact with physical berth constraints. Its results apply to the supplied data and assumptions; they should not be read as general performance guarantees. Natural extensions within the same scheduling framework include supplying vessel-specific dimensions, testing additional arrival patterns, or adding operational constraints to the input data and compatibility rules.